import os
import re
from pathlib import Path
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ==========================================
# Global Mangrove Watch (GMW v4.1) — Reprojection & Area Calculation
# ==========================================
import os
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import Resampling, calculate_default_transform, reproject
import seaborn as sns

# ==========================================
# 1. DIRECTORY & CONSTANT CONFIGURATION
# ==========================================
# Override these paths via environment variables or update them for your local machine
INPUT_DIR = Path(os.getenv("GMW_INPUT_DIR", r"D:\MANGROVES\GMW\V4\gmw_mng_ext_v4112_stack"))
PROJ_DIR = Path(os.getenv("GMW_PROJ_DIR", INPUT_DIR / "reprojected_54034"))
OUT_DIR = Path(os.getenv("GMW_OUT_DIR", r"D:\MANGROVES\V4"))
GEOJSON_TILES_PATH = Path(
    os.getenv("GMW_TILES_GEOJSON", r"D:\MANGROVES\GMW\V4\gmw_v4112_tiles.geojson")
)

PROJ_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CRS = "ESRI:54034"

# Regex to extract GMW v4 tile ID (e.g., GMW_N00E008_v4112_mng_ext.tif -> N00E008)
FILE_PATTERN = re.compile(
    r"^GMW_([NS]\d{2,3}[EW]\d{3})_v4112_mng_ext\.tif$", re.IGNORECASE
)

# Band-to-year mapping: Band 1 = 1985, Band 41 = 2025
YEARS = list(range(1985, 2026))

# ==========================================
# 2. GEOJSON REPROJECTION & CENTROID EXTRACTION
# ==========================================
print(f"Loading and reprojecting tile grid: {GEOJSON_TILES_PATH}...")
gdf_tiles = gpd.read_file(GEOJSON_TILES_PATH)

# Reproject vector layer to ESRI:54034 (meters) for planar area/centroid accuracy
gdf_tiles_proj = gdf_tiles.to_crs(TARGET_CRS)

# Compute centroids in planar ESRI:54034 coordinates, then convert to WGS84 (EPSG:4326)
centroids_wgs84 = gdf_tiles_proj.geometry.centroid.to_crs(epsg=4326)

tile_coords = {
    row["tilename"]: {
        "center_lat": float(centroids_wgs84.iloc[idx].y),
        "center_lon": float(centroids_wgs84.iloc[idx].x),
    }
    for idx, row in gdf_tiles_proj.iterrows()
}

# Save reprojected tile grid GeoJSON
geojson_proj_path = OUT_DIR / "gmw_v4112_tiles_54034.geojson"
gdf_tiles_proj.to_file(geojson_proj_path, driver="GeoJSON")
print(f"Saved reprojected GeoJSON: {geojson_proj_path}")

# ==========================================
# 3. MULTI-BAND REPROJECTION & AREA CALCULATION
# ==========================================
results = []
tif_files = sorted(list(INPUT_DIR.glob("*.tif")) + list(INPUT_DIR.glob("*.tiff")))

print(f"Starting processing of {len(tif_files)} GMW v4.1 raster file(s)...")

for tif_path in tif_files:
    match = FILE_PATTERN.match(tif_path.name)
    if not match:
        continue

    tile_id = match.group(1).upper()

    # Retrieve tile center coordinates from GeoJSON lookup
    if tile_id in tile_coords:
        center_lat = tile_coords[tile_id]["center_lat"]
        center_lon = tile_coords[tile_id]["center_lon"]
    else:
        # Fallback if the tile ID is missing from the GeoJSON grid
        center_lat, center_lon = 0.0, 0.0

    print(f"Processing tile {tile_id} (Lat {center_lat:.2f}°, Lon {center_lon:.2f}°)")

    reprojected_path = PROJ_DIR / f"{tif_path.stem}_54034.tif"

    with rasterio.open(tif_path) as src:
        n_bands = src.count

        # Compute target transform and dimensions for ESRI:54034
        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height,
            "count": n_bands,
            "nodata": 0,
        })

        data_reprojected = np.zeros((n_bands, height, width), dtype=src.dtypes[0])

        # Nearest-neighbor resampling preserves discrete classification values (0/1)
        for band_idx in range(1, n_bands + 1):
            reproject(
                source=rasterio.band(src, band_idx),
                destination=data_reprojected[band_idx - 1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.nearest,
            )

        # Write reprojected multi-band GeoTIFF
        with rasterio.open(reprojected_path, "w", **kwargs) as dst:
            dst.write(data_reprojected)

        # Pixel area in m² within the equal-area ESRI:54034 projection
        pixel_area_m2 = abs(transform.a) * abs(transform.e)

        # Extract mangrove pixel count and area per year (band by band)
        for band_idx in range(1, n_bands + 1):
            year = YEARS[band_idx - 1] if band_idx <= len(YEARS) else 1985 + (band_idx - 1)

            band_data = data_reprojected[band_idx - 1]
            pixel_count = int(np.count_nonzero(band_data == 1))
            area_m2 = pixel_count * pixel_area_m2
            area_km2 = area_m2 / 1_000_000.0

            results.append({
                "tile_id": tile_id,
                "year": int(year),
                "center_lat": round(center_lat, 4),
                "center_lon": round(center_lon, 4),
                "pixel_count": pixel_count,
                "area_km2": round(area_km2, 4),
                "file_name": tif_path.name,
            })

# ==========================================
# 4. EXPORT RESULTS TO CSV
# ==========================================
df_results = pd.DataFrame(results)
df_results.sort_values(by=["center_lon", "center_lat", "year"], inplace=True)
csv_output_path = OUT_DIR / "gmwv4_surfaces_annual_1deg_tiles.csv"
df_results.to_csv(csv_output_path, index=False)
print(f"\nResults exported to: {csv_output_path}")